# ContractSense Semantic Copilot on Lightning AI

This notebook is the single-file runner for the upgraded pipeline.

It does four things in one flow:
1. installs requirements and sets L4-safe resource knobs,
2. runs the semantic DPO training pipeline,
3. generates baseline vs generator vs DPO evaluation artifacts,
4. pushes the merged model to Hugging Face and writes a run summary file.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "requirements_pipeline.txt").exists() and REPO_ROOT.parent != REPO_ROOT:
    REPO_ROOT = REPO_ROOT.parent

if not (REPO_ROOT / "requirements_pipeline.txt").exists():
    raise FileNotFoundError("Could not find requirements_pipeline.txt. Open this notebook from inside the ContractSense repo.")

os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))
sys.path.insert(0, str(REPO_ROOT / "scripts"))

print(f"Repo root: {REPO_ROOT}")


In [ ]:
import subprocess

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", str(REPO_ROOT / "requirements_pipeline.txt")],
    check=True,
)
print("Requirements installed.")


In [ ]:
import multiprocessing
import torch

cpu_count = multiprocessing.cpu_count()
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
os.environ.setdefault("OMP_NUM_THREADS", str(min(8, cpu_count)))
os.environ.setdefault("MKL_NUM_THREADS", str(min(8, cpu_count)))
os.environ.setdefault("DATALOADER_WORKERS", str(min(4, max(2, cpu_count // 2))))

# Stable L4 defaults for 7B DPO in 4-bit.
os.environ.setdefault("PER_DEVICE_BATCH_SIZE", "4")
os.environ.setdefault("GRAD_ACCUM", "4")
os.environ.setdefault("SAVE_STEPS", "50")
os.environ.setdefault("RESUME_FROM_CHECKPOINT", "0")

torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print(f"Python: {sys.version.split()[0]}")
print(f"CPU cores visible: {cpu_count}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    props = torch.cuda.get_device_properties(0)
    print(f"GPU: {gpu_name}")
    print(f"VRAM GB: {props.total_memory / (1024**3):.2f}")
    if "L4" in gpu_name.upper():
        print("Runtime profile: NVIDIA L4 detected; high-throughput defaults are active.")
    else:
        print("Runtime profile: non-L4 CUDA GPU detected; adjust batch env vars if needed.")
print("Lightning optimization env vars configured.")


In [ ]:
from huggingface_hub import login

hf_token = os.getenv("HF_TOKEN") or os.getenv("HUGGINGFACEHUB_API_TOKEN") or os.getenv("HUGGINGFACE_HUB_TOKEN")
if hf_token:
    login(token=hf_token, add_to_git_credential=True)
    print("Hugging Face login refreshed from environment token.")
else:
    print("No HF token found in environment. If Lightning is already logged in, push_to_hub will still work.")


In [ ]:
import shutil

FRESH_RUN = True
OUTPUT_DIR = REPO_ROOT / "grounded_dpo_model_v2"

if FRESH_RUN and OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)
    print(f"Removed stale output dir: {OUTPUT_DIR}")
else:
    print(f"Keeping output dir: {OUTPUT_DIR}")


In [ ]:
import json
import lightning_train_v2 as train_mod

summary = train_mod.run_full_pipeline(output_dir=str(OUTPUT_DIR), push=True)

print("\nTraining run complete.")
print(json.dumps({
    "dataset_size": summary["dataset_size"],
    "hf_repo": summary["hf_repo"],
    "eval_results": summary["eval_results"],
    "summary_path": summary["summary_path"],
}, indent=2))


In [ ]:
from pathlib import Path
import pandas as pd

summary_path = Path(summary["summary_path"])
comparison_csv = OUTPUT_DIR / "images" / "model_comparison_metrics.csv"
comparison_json = OUTPUT_DIR / "images" / "model_comparison_metrics.json"

print(f"Run summary file: {summary_path}")
print(f"Comparison CSV: {comparison_csv}")
print(f"Comparison JSON: {comparison_json}")

if comparison_csv.exists():
    display(pd.read_csv(comparison_csv))
else:
    print("Comparison CSV was not generated.")


## Artifacts to Expect

After the notebook finishes, check these paths:

- `grounded_dpo_model_v2/final/`
- `grounded_dpo_model_v2/eval_results.json`
- `grounded_dpo_model_v2/precision_pipeline_metrics.json`
- `grounded_dpo_model_v2/images/model_comparison_metrics.csv`
- `grounded_dpo_model_v2/images/model_comparison_metrics.json`
- `grounded_dpo_model_v2/images/baseline_generator_dpo_comparison.png`
- `grounded_dpo_model_v2/images/hallucination_rate_comparison.png`
- `grounded_dpo_model_v2/run_summary.md`
